<a href="https://colab.research.google.com/github/JuliusdeGroot/gpu/blob/main/gpu_test.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# GPU Programming

To use CUDA, you need to enable a GPU runtime. Here's how you can change your Colab runtime to use a GPU:

- Go to the "Runtime" menu at the top of the Colab interface.
- Select "Change runtime type".
- In the "Hardware accelerator" dropdown menu, choose a "GPU".
- Click "Save".

Now run the following blocks, by clicking the 'Play' icon or setting focus and pressing Shift-Enter, to set things up and run the add_vec_gpu3 example.

In [5]:
# Install NVIDIA CUDA Toolkit, this can take a while.
!apt-get -qq install nvidia-cuda-toolkit
print('NVIDIA CUDA Toolkit installation complete. Please run the next cell to verify.')

Extracting templates from packages: 100%
Preconfiguring packages ...
Selecting previously unselected package libdebuginfod-common.
(Reading database ... 122412 files and directories currently installed.)
Preparing to unpack .../00-libdebuginfod-common_0.186-1ubuntu0.1_all.deb ...
Unpacking libdebuginfod-common (0.186-1ubuntu0.1) ...
Selecting previously unselected package libatspi2.0-0:amd64.
Preparing to unpack .../01-libatspi2.0-0_2.44.0-3_amd64.deb ...
Unpacking libatspi2.0-0:amd64 (2.44.0-3) ...
Selecting previously unselected package libxtst6:amd64.
Preparing to unpack .../02-libxtst6_2%3a1.2.3-1build4_amd64.deb ...
Unpacking libxtst6:amd64 (2:1.2.3-1build4) ...
Selecting previously unselected package session-migration.
Preparing to unpack .../03-session-migration_0.3.6_amd64.deb ...
Unpacking session-migration (0.3.6) ...
Selecting previously unselected package gsettings-desktop-schemas.
Preparing to unpack .../04-gsettings-desktop-schemas_42.0-1ubuntu1_all.deb ...
Unpacking gset

In [6]:
# Check the nvcc version to confirm CUDA installation
!nvcc --version

nvcc: NVIDIA (R) Cuda compiler driver
Copyright (c) 2005-2025 NVIDIA Corporation
Built on Fri_Feb_21_20:23:50_PST_2025
Cuda compilation tools, release 12.8, V12.8.93
Build cuda_12.8.r12.8/compiler.35583870_0


### Run the following blocks to add the files

- utils.h         : with some vector helper functions
- gpu_utils.h     : with some CUDA helper functions
- add_vec_gpu3.cu : the vector addition example  


In [7]:
%%writefile utils.h
#ifndef __UTILS_H__
#define __UTILS_H__

#include <vector>
#include <random>
#include <iostream>
#include <cfloat>
using namespace std;

vector<float> get_random_vector(int n) {
    vector<float> v(n);
    for (int i = 0; i < n; i++)
        v[i] = static_cast<float>(rand()) / RAND_MAX;
    return v;
}

ostream& operator<<(ostream& os, const vector<float>& v) {
    for (size_t i = 0; i < v.size(); i++)
        os << v[i] << ' ';
    return os;
}

#endif

Writing utils.h


In [8]:
%%writefile gpu_utils.h
#ifndef __GPU_UTILS_H__
#define __GPU_UTILS_H__

#include <cuda_runtime.h>
#include <iostream>
#include <stdio.h>
#include "utils.h"
#include <chrono>

using namespace std;
using namespace chrono;

static void HandleError( cudaError_t err,
                         const char *file,
                         int line ) {
    if (err != cudaSuccess) {
        printf( "%s in %s at line %d\n", cudaGetErrorString( err ),
                file, line );
        exit( EXIT_FAILURE );
    }
}
#define HANDLE_ERROR( err ) (HandleError( err, __FILE__, __LINE__ ))

int get_cuda_device_count() {
    int count;
    HANDLE_ERROR(cudaGetDeviceCount(&count));
    return count;
}

cudaDeviceProp get_cuda_device_properties(int device_id) {
    cudaDeviceProp prop;
    HANDLE_ERROR(cudaGetDeviceProperties(&prop, device_id));
    return prop;
}

void print_cuda_device_properties(const cudaDeviceProp& prop) {
    std::cout<< "Device  prop.name: " << prop.name << std::endl;
    std::cout<< "  Compute capability  prop.major: " << prop.major << ", prop.minor: " << prop.minor << std::endl;
    std::cout<< "  Clock rate  prop.clockRate: " << prop.clockRate << std::endl;
    std::cout<< "  Multiprocessor count  prop.multiProcessorCount: " << prop.multiProcessorCount << std::endl;
    std::cout<< "  Total global memory  prop.totalGlobalMem: " << prop.totalGlobalMem << " bytes" << std::endl;
    std::cout<< "  Total constant memory  prop.totalConstMem: " << prop.totalConstMem << " bytes" << std::endl;
    std::cout<< "  Shared memory per block  prop.sharedMemPerBlock: " << prop.sharedMemPerBlock << " bytes" << std::endl;
    std::cout<< "  Shared memory per multiprocessor  prop.sharedMemPerMultiprocessor: " << prop.sharedMemPerMultiprocessor << " bytes" << std::endl;
    std::cout<< "  Registers per block  prop.regsPerBlock: " << prop.regsPerBlock << std::endl;
    std::cout<< "  Registers per multiprocessor  prop.regsPerMultiprocessor: " << prop.regsPerMultiprocessor << std::endl;
    std::cout<< "  Warp size  prop.warpSize: " << prop.warpSize << std::endl;
    std::cout<< "  Max threads per block  prop.maxThreadsPerBlock: " << prop.maxThreadsPerBlock << std::endl;
    std::cout<< "  Max threads per multiprocessor  prop.maxThreadsPerMultiProcessor: " << prop.maxThreadsPerMultiProcessor << std::endl;
    std::cout<< "  Max thread dimensions  prop.maxThreadsDim: (" << prop.maxThreadsDim[0] << ", " << prop.maxThreadsDim[1] << ", " << prop.maxThreadsDim[2] << ")" << std::endl;
    std::cout<< "  Max grid dimensions  prop.maxGridSize: (" << prop.maxGridSize[0] << ", " << prop.maxGridSize[1] << ", " << prop.maxGridSize[2] << ")" << std::endl;
    std::cout<< "  Concurrent kernels  prop.concurrentKernels: " << prop.concurrentKernels << std::endl;
    std::cout<< "  Async engine count  prop.asyncEngineCount: " << prop.asyncEngineCount << std::endl;
    std::cout<< "  Kernel execution timeout enabled  prop.kernelExecTimeoutEnabled: " << prop.kernelExecTimeoutEnabled << std::endl;
}


#endif

Writing gpu_utils.h


In [9]:
%%writefile add_vec_gpu3.cu
#include "gpu_utils.h"

__global__ void add_vec_gpu(const float* d_v1, const float* d_v2, float* d_add, int n) {
    for (int i = threadIdx.x + blockIdx.x * blockDim.x;
        i < n;
        i += blockDim.x * gridDim.x)
        d_add[i] = d_v1[i] + d_v2[i];
}

int main() {
    //srand(time(nullptr));
    const int n = 18;
    vector<float> h_v1 = get_random_vector(n);
    vector<float> h_v2 = get_random_vector(n);
    vector<float> h_add(n);

    cout << "v1: " << h_v1 << endl;
    cout << "v2: " << h_v2 << endl;

    float *d_v1, *d_v2, *d_add;
    // Allocate device memory
    HANDLE_ERROR(cudaMalloc((void**)&d_v1, sizeof(float) * n));
    HANDLE_ERROR(cudaMalloc((void**)&d_v2, sizeof(float) * n));
    HANDLE_ERROR(cudaMalloc((void**)&d_add, sizeof(float) * n));

    // Transfer data from host to device memory
    HANDLE_ERROR(cudaMemcpy(d_v1, h_v1.data(), sizeof(float) * n, cudaMemcpyHostToDevice));
    HANDLE_ERROR(cudaMemcpy(d_v2, h_v2.data(), sizeof(float) * n, cudaMemcpyHostToDevice));

    add_vec_gpu<<<2, 4>>>(d_v1, d_v2, d_add, n);
    HANDLE_ERROR(cudaGetLastError());
    HANDLE_ERROR(cudaDeviceSynchronize());

    // Transfer data back to host memory
    HANDLE_ERROR(cudaMemcpy(h_add.data(), d_add, sizeof(float) * n, cudaMemcpyDeviceToHost));

    // Deallocate device memory
    HANDLE_ERROR(cudaFree(d_v1));
    HANDLE_ERROR(cudaFree(d_v2));
    HANDLE_ERROR(cudaFree(d_add));

    cout << "addition: " << h_add << endl;
}

Writing add_vec_gpu3.cu


In [10]:
# Now compile the CUDA program using nvcc
# -o specifies the output executable name
!nvcc add_vec_gpu3.cu -o add_vec_gpu3

nvcc warning : Support for offline compilation for architectures prior to '<compute/sm/lto>_75' will be removed in a future release (Use -Wno-deprecated-gpu-targets to suppress warning).


In [11]:
# Run the compiled CUDA program
!./add_vec_gpu3

v1: 0.840188 0.394383 0.783099 0.79844 0.911647 0.197551 0.335223 0.76823 0.277775 0.55397 0.477397 0.628871 0.364784 0.513401 0.95223 0.916195 0.635712 0.717297 
v2: 0.141603 0.606969 0.0163006 0.242887 0.137232 0.804177 0.156679 0.400944 0.12979 0.108809 0.998924 0.218257 0.512932 0.839112 0.61264 0.296032 0.637552 0.524287 
addition: 0.98179 1.00135 0.7994 1.04133 1.04888 1.00173 0.491902 1.16917 0.407565 0.662779 1.47632 0.847128 0.877717 1.35251 1.56487 1.21223 1.27326 1.24158 


In [12]:
%%writefile normalize_vec_gpu.cu

#include "gpu_utils.h"
#include <cmath>
#include <chrono>

using namespace std;
using namespace chrono;

__global__ void normalize_vec_gpu(float* d_v,
                                  float length,
                                  int n) {

    for (int i = threadIdx.x + blockIdx.x * blockDim.x;
         i < n;
         i += blockDim.x * gridDim.x) {

        d_v[i] /= length;
    }
}

float square_length_cpu(const vector<float>& v) {

    float sum = 0;

    for (int i = 0; i < v.size(); i++)
        sum += v[i] * v[i];

    return sum;
}

int main() {

    const int n = 1000000;

    // Original vector
    vector<float> h_v = get_random_vector(n);

    // ================= CPU VERSION =================

    vector<float> cpu_v = h_v;

    // Compute length BEFORE timing
    float cpu_length =
        sqrt(square_length_cpu(cpu_v));

    auto cpu_start =
        high_resolution_clock::now();

    // Only normalize during timing
    for (int i = 0; i < n; i++)
        cpu_v[i] /= cpu_length;

    auto cpu_end =
        high_resolution_clock::now();

    auto cpu_duration =
        duration_cast<microseconds>(
            cpu_end - cpu_start
        );

    cout << "CPU normalization time: "
         << cpu_duration.count()
         << " microseconds"
         << endl;

    // ================= GPU VERSION =================

    float* d_v;

    HANDLE_ERROR(cudaMalloc(
        (void**)&d_v,
        sizeof(float) * n
    ));

    HANDLE_ERROR(cudaMemcpy(
        d_v,
        h_v.data(),
        sizeof(float) * n,
        cudaMemcpyHostToDevice
    ));

    // Compute same length BEFORE timing
    float gpu_length =
        sqrt(square_length_cpu(h_v));

    auto gpu_start =
        high_resolution_clock::now();

    // Only normalize during timing
    normalize_vec_gpu<<<256,256>>>(
        d_v,
        gpu_length,
        n
    );

    HANDLE_ERROR(cudaGetLastError());

    HANDLE_ERROR(cudaDeviceSynchronize());

    auto gpu_end =
        high_resolution_clock::now();

    auto gpu_duration =
        duration_cast<microseconds>(
            gpu_end - gpu_start
        );

    cout << "GPU normalization time: "
         << gpu_duration.count()
         << " microseconds"
         << endl;

    // Copy result back
    HANDLE_ERROR(cudaMemcpy(
        h_v.data(),
        d_v,
        sizeof(float) * n,
        cudaMemcpyDeviceToHost
    ));

    HANDLE_ERROR(cudaFree(d_v));

    // Verify normalization
    float new_length =
        sqrt(square_length_cpu(h_v));

    cout << "length after normalization: "
         << new_length
         << endl;
}

Writing normalize_vec_gpu.cu


In [13]:
!nvcc normalize_vec_gpu.cu -o normalize_vec_gpu

nvcc warning : Support for offline compilation for architectures prior to '<compute/sm/lto>_75' will be removed in a future release (Use -Wno-deprecated-gpu-targets to suppress warning).


In [14]:
!./normalize_vec_gpu

CPU normalization time: 2915 microseconds
GPU normalization time: 22845 microseconds
length after normalization: 1.00002


I am only measuring kernel execution time. This excludes memory transfers and GPU memory allocation, which are not part of the compute kernel itself.

In [15]:
%%writefile max_vec_gpu.cu

#include "gpu_utils.h"
#include <cmath>
#include <float.h>
#include <tuple>
#include <chrono>

using namespace std;
using namespace chrono;

tuple<float, int> max_element_cpu(const vector<float>& v) {

    float max_el = -FLT_MAX;
    int max_idx = 0;

    for (size_t i = 0; i < v.size(); i++) {
        if (v[i] > max_el) {
            max_el = v[i];
            max_idx = i;
        }
    }

    return {max_el, max_idx};
}

__global__ void max_vec_gpu(const float* d_v,
                            float* d_block_max,
                            int* d_block_idx,
                            int n) {

    int tid = threadIdx.x;
    int idx = threadIdx.x + blockIdx.x * blockDim.x;

    extern __shared__ float smax[];
    int* sidx = (int*)&smax[blockDim.x];

    float max_val = -FLT_MAX;
    int max_i = -1;

    if (idx < n) {
        max_val = d_v[idx];
        max_i = idx;
    }

    smax[tid] = max_val;
    sidx[tid] = max_i;

  // Ga pas verder als allthe threads in het block hun waardes op shared memory hebben gezet
    __syncthreads();

    // block reduction
    for (int stride = blockDim.x / 2; stride > 0; stride >>= 1) {
        if (tid < stride) {
            if (smax[tid + stride] > smax[tid]) {
                smax[tid] = smax[tid + stride];
                sidx[tid] = sidx[tid + stride];
            }
        }
        __syncthreads();
    }

    if (tid == 0) {
        d_block_max[blockIdx.x] = smax[0];
        d_block_idx[blockIdx.x] = sidx[0];
    }
}

// ---------------- MAIN ----------------
int main() {

    const int n = 1000000;
    vector<float> h_v = get_random_vector(n);

    // ---------------- CPU ----------------
    auto cpu_start = high_resolution_clock::now();

    auto [cpu_max, cpu_idx] = max_element_cpu(h_v);

    auto cpu_end = high_resolution_clock::now();

    cout << "CPU time: "
         << duration_cast<microseconds>(cpu_end - cpu_start).count()
         << " us\n";

    // ---------------- GPU ----------------
    float *d_v, *d_block_max;
    int *d_block_idx;

    int threads = 256;
    int blocks = (n + threads - 1) / threads;

    HANDLE_ERROR(cudaMalloc(&d_v, sizeof(float) * n));
    HANDLE_ERROR(cudaMalloc(&d_block_max, sizeof(float) * blocks));
    HANDLE_ERROR(cudaMalloc(&d_block_idx, sizeof(int) * blocks));

    HANDLE_ERROR(cudaMemcpy(d_v, h_v.data(),
                             sizeof(float) * n,
                             cudaMemcpyHostToDevice));

    auto gpu_start = high_resolution_clock::now();

    max_vec_gpu<<<blocks, threads,
        threads * (sizeof(float) + sizeof(int))>>>(
        d_v, d_block_max, d_block_idx, n);

    HANDLE_ERROR(cudaDeviceSynchronize());

    vector<float> h_block_max(blocks);
    vector<int> h_block_idx(blocks);

    HANDLE_ERROR(cudaMemcpy(h_block_max.data(),
                            d_block_max,
                            sizeof(float) * blocks,
                            cudaMemcpyDeviceToHost));

    // final CPU step (small)
    float gpu_max = -FLT_MAX;
    int gpu_idx = 0;

    for (int i = 0; i < blocks; i++) {
        if (h_block_max[i] > gpu_max) {
            gpu_max = h_block_max[i];
            gpu_idx = h_block_idx[i];
        }
    }

    auto gpu_end = high_resolution_clock::now();

    cout << "GPU time: "
         << duration_cast<microseconds>(gpu_end - gpu_start).count()
         << " us\n";

    cudaFree(d_v);
    cudaFree(d_block_max);
    cudaFree(d_block_idx);
}


Writing max_vec_gpu.cu


In [16]:
!nvcc max_vec_gpu.cu -o max_vec_gpu

nvcc warning : Support for offline compilation for architectures prior to '<compute/sm/lto>_75' will be removed in a future release (Use -Wno-deprecated-gpu-targets to suppress warning).
max_vec_gpu.cu(118): warning #550-D: variable "gpu_idx" was set but never used
      int gpu_idx = 0;
          ^

Remark: The warnings can be suppressed with "-diag-suppress <warning-number>"



In [17]:
!./max_vec_gpu

CPU time: 4813 us
GPU time: 17205 us


In [21]:
%%writefile mat_vec_product_gpu.cu

#include "gpu_utils.h"
#include <chrono>

using namespace std;
using namespace chrono;


// ================= CPU =================

vector<float> mat_vec_cpu(const vector<float>& A,
                          const vector<float>& x,
                          int rows,
                          int cols) {

    vector<float> y(rows, 0);

    for (int i = 0; i < rows; i++) {

        float sum = 0;

        for (int j = 0; j < cols; j++) {
            sum += A[i * cols + j] * x[j];
        }

        y[i] = sum;
    }

    return y;
}


// ================= GPU =================

__global__ void mat_vec_gpu(const float* A,
                            const float* x,
                            float* y,
                            int rows,
                            int cols) {

    int row =
        blockIdx.x * blockDim.x + threadIdx.x;

    if (row < rows) {

        float sum = 0;

        for (int j = 0; j < cols; j++) {
            sum += A[row * cols + j] * x[j];
        }

        y[row] = sum;
    }
}


// ================= MAIN =================

int main() {

    int rows = 1000;
    int cols = 1000;

    vector<float> h_A =
        get_random_vector(rows * cols);

    vector<float> h_x =
        get_random_vector(cols);

    vector<float> h_y(rows);

    // ================= CPU =================

    auto cpu_start =
        high_resolution_clock::now();

    vector<float> cpu_y =
        mat_vec_cpu(h_A, h_x, rows, cols);

    auto cpu_end =
        high_resolution_clock::now();

    cout << "CPU time: "
         << duration_cast<microseconds>(
                cpu_end - cpu_start
            ).count()
         << " us" << endl;


    // ================= GPU =================

    float *d_A, *d_x, *d_y;

    HANDLE_ERROR(cudaMalloc(
        (void**)&d_A,
        sizeof(float) * rows * cols
    ));

    HANDLE_ERROR(cudaMalloc(
        (void**)&d_x,
        sizeof(float) * cols
    ));

    HANDLE_ERROR(cudaMalloc(
        (void**)&d_y,
        sizeof(float) * rows
    ));

    HANDLE_ERROR(cudaMemcpy(
        d_A,
        h_A.data(),
        sizeof(float) * rows * cols,
        cudaMemcpyHostToDevice
    ));

    HANDLE_ERROR(cudaMemcpy(
        d_x,
        h_x.data(),
        sizeof(float) * cols,
        cudaMemcpyHostToDevice
    ));

    int threads = 256;

    int blocks =
        (rows + threads - 1) / threads;

    auto gpu_start =
        high_resolution_clock::now();

    mat_vec_gpu<<<blocks, threads>>>(
        d_A,
        d_x,
        d_y,
        rows,
        cols
    );

    HANDLE_ERROR(cudaGetLastError());

    HANDLE_ERROR(cudaDeviceSynchronize());

    auto gpu_end =
        high_resolution_clock::now();

    cout << "GPU time: "
         << duration_cast<microseconds>(
                gpu_end - gpu_start
            ).count()
         << " us" << endl;


    HANDLE_ERROR(cudaMemcpy(
        h_y.data(),
        d_y,
        sizeof(float) * rows,
        cudaMemcpyDeviceToHost
    ));

    HANDLE_ERROR(cudaFree(d_A));
    HANDLE_ERROR(cudaFree(d_x));
    HANDLE_ERROR(cudaFree(d_y));
}

Overwriting mat_vec_product_gpu.cu


In [22]:
!nvcc mat_vec_product_gpu.cu -o mat_vec_product_gpu

nvcc warning : Support for offline compilation for architectures prior to '<compute/sm/lto>_75' will be removed in a future release (Use -Wno-deprecated-gpu-targets to suppress warning).


In [23]:
!./max_vec_gpu

CPU time: 4795 us
GPU time: 378 us


In [38]:
%%writefile radix_sort_gpu.cu

#include "gpu_utils.h"
#include <chrono>

using namespace std;
using namespace chrono;

const int BASE = 10;


// ================= GPU KERNEL =================

// extract digits in parallel
__global__ void extract_digits_gpu(
    const int* d_data,
    int* d_digits,
    int exp,
    int n) {

    int i =
        blockIdx.x * blockDim.x + threadIdx.x;

    if (i < n) {

        d_digits[i] =
            (d_data[i] / exp) % BASE;
    }
}


// ================= MAIN =================

int main() {

    const int n = 1000000;

    vector<int> h_data(n);

    for (int i = 0; i < n; i++)
        h_data[i] = rand() % 1000000;

    int *d_data, *d_digits;

    HANDLE_ERROR(cudaMalloc(
        &d_data,
        sizeof(int) * n
    ));

    HANDLE_ERROR(cudaMalloc(
        &d_digits,
        sizeof(int) * n
    ));

    HANDLE_ERROR(cudaMemcpy(
        d_data,
        h_data.data(),
        sizeof(int) * n,
        cudaMemcpyHostToDevice
    ));

    int threads = 256;

    int blocks =
        (n + threads - 1) / threads;

    auto gpu_start =
        high_resolution_clock::now();

    // extract least significant digit
    extract_digits_gpu<<<blocks, threads>>>(
        d_data,
        d_digits,
        1,
        n
    );

    HANDLE_ERROR(cudaGetLastError());

    HANDLE_ERROR(cudaDeviceSynchronize());

    auto gpu_end =
        high_resolution_clock::now();

  cout << "GPU digit extraction time: "
      << duration_cast<microseconds>(
              gpu_end - gpu_start
          ).count()
      << " us"
      << endl;

    HANDLE_ERROR(cudaFree(d_data));
    HANDLE_ERROR(cudaFree(d_digits));
}

Overwriting radix_sort_gpu.cu


In [34]:
!nvcc radix_sort_gpu.cu -o radix_sort_gpu

nvcc warning : Support for offline compilation for architectures prior to '<compute/sm/lto>_75' will be removed in a future release (Use -Wno-deprecated-gpu-targets to suppress warning).


In [35]:
!./radix_sort_gpu

GPU digit extraction time: 230 us


In [39]:
import random
import time

BASE = 10


def counting_sort(data, data_new, digit):
    counts = [0] * BASE
    exp = BASE ** digit

    for d in data:
        counts[(d // exp) % BASE] += 1

    for i in range(1, len(counts)):
        counts[i] += counts[i - 1]

    for i in range(len(data) - 1, -1, -1):
        d = (data[i] // exp) % BASE
        counts[d] -= 1
        data_new[counts[d]] = data[i]


def radix_sort(data, nr_digits):
    data_new = [0] * len(data)

    for d in range(nr_digits):
        counting_sort(data, data_new, d)
        data, data_new = data_new, data

    return data


def get_random_data(n, nr_digits):
    range_end = BASE ** nr_digits
    return [random.randrange(range_end) for _ in range(n)]


# ================= MAIN =================

n = 1000000
nr_digits = 6

data = get_random_data(n, nr_digits)

start = time.perf_counter()

data_sorted = radix_sort(data, nr_digits)

end = time.perf_counter()

elapsed_us = (end - start) * 1_000_000

print(f"CPU radix sort time: {elapsed_us:.0f} us")

CPU radix sort time: 2497041 us
